# 02 — Análise Exploratória de Dados (EDA)

Neste notebook realizamos a análise exploratória completa do dataset de controle de qualidade,
buscando entender a distribuição dos parâmetros, identificar padrões e correlações,
e caracterizar o perfil de lotes aprovados vs. reprovados.

**Etapas:**
1. Carregamento e visão geral dos dados
2. Distribuição das variáveis numéricas
3. Análise por status do lote
4. Correlações
5. Análise por turno
6. Tendência temporal

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/raw/lotes_qc.csv', parse_dates=['data_producao'])
print(f"Shape: {df.shape}")
df.head()

## 1. Visão Geral dos Dados

In [ ]:
print("=== Informações gerais ===")
df.info()
print("\n=== Valores ausentes ===")
print(df.isnull().sum())
print("\n=== Estatísticas descritivas ===")
df.describe().round(2)

In [ ]:
# Distribuição da variável alvo
fig, ax = plt.subplots(figsize=(5, 4))
contagem = df['status_lote'].value_counts()
cores = ['#2ecc71', '#e74c3c']
contagem.plot(kind='bar', ax=ax, color=cores, edgecolor='white', width=0.5)
ax.set_title('Distribuição do Status dos Lotes', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Quantidade de Lotes')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(contagem):
    ax.text(i, v + 3, f'{v} ({v/len(df):.1%})', ha='center', fontweight='bold')
plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/01_distribuicao_status.png')
plt.show()

## 2. Distribuição dos Parâmetros por Status do Lote

In [ ]:
variaveis = ['temp_processo', 'umidade_relativa', 'pressao_compressao',
             'dureza_media', 'friabilidade', 'peso_medio', 'dissolucao']

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, var in enumerate(variaveis):
    for status, cor in [('APROVADO', '#2ecc71'), ('REPROVADO', '#e74c3c')]:
        subset = df[df['status_lote'] == status][var]
        axes[i].hist(subset, bins=25, alpha=0.6, color=cor, label=status, edgecolor='white')
    axes[i].set_title(var.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

# Desligar subplots extras
for j in range(len(variaveis), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição dos Parâmetros por Status do Lote', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/02_distribuicao_parametros.png', bbox_inches='tight')
plt.show()

## 3. Matriz de Correlação

In [ ]:
# Adiciona coluna numérica para correlação com status
df['reprovado'] = (df['status_lote'] == 'REPROVADO').astype(int)

corr = df[variaveis + ['reprovado']].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlação — Parâmetros de Processo e Qualidade', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_correlacao.png')
plt.show()

print("\nCorrelação com 'reprovado':")
print(corr['reprovado'].drop('reprovado').sort_values(key=abs, ascending=False).round(3))

## 4. Taxa de Reprovação por Turno

In [ ]:
turno_stats = df.groupby('turno').agg(
    total=('lote', 'count'),
    reprovados=('reprovado', 'sum')
).assign(taxa_reprovacao=lambda x: x['reprovados'] / x['total'] * 100).round(1)

print(turno_stats)

fig, ax = plt.subplots(figsize=(6, 4))
turno_stats['taxa_reprovacao'].plot(kind='bar', ax=ax, color='#3498db', edgecolor='white', width=0.5)
ax.set_title('Taxa de Reprovação por Turno (%)', fontweight='bold')
ax.set_xlabel('Turno')
ax.set_ylabel('Taxa de Reprovação (%)')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(turno_stats['taxa_reprovacao']):
    ax.text(i, v + 0.2, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_reprovacao_turno.png')
plt.show()

## 5. Tendência Temporal — Taxa de Reprovação Mensal

In [ ]:
df['mes'] = df['data_producao'].dt.to_period('M')
mensal = df.groupby('mes').agg(
    total=('lote', 'count'),
    reprovados=('reprovado', 'sum')
).assign(taxa=lambda x: x['reprovados'] / x['total'] * 100)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mensal.index.astype(str), mensal['taxa'], marker='o', color='#e74c3c', linewidth=2)
ax.axhline(mensal['taxa'].mean(), color='gray', linestyle='--', label=f'Média: {mensal["taxa"].mean():.1f}%')
ax.set_title('Taxa de Reprovação Mensal (%)', fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Taxa de Reprovação (%)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/05_tendencia_mensal.png')
plt.show()

## Conclusões da EDA

- **Friabilidade e dissolução** apresentam as maiores correlações com reprovação de lotes
- **Pressão de compressão** influencia diretamente dureza e friabilidade — variável de processo crítica
- **Turnos** apresentam taxas de reprovação similares, sem desvio significativo entre equipes
- A tendência temporal será investigada no notebook de modelagem

➡️ Próximo passo: **Notebook 03 — Modelo Preditivo**